In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1'

import torch  # Torch must claim OpenMP bindings first
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import gc
import warnings
warnings.filterwarnings("ignore")

In [2]:
data=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/AML - GNN/data/processed/final_vectors.parquet')
graph=torch.load('/Users/rageshwer/Goal ML/Projects/AML - GNN/data/graphs/elliptic_graph_processed.pt',weights_only=False)
labelled_mask=((graph.y==0) | (graph.y==1))
mask=labelled_mask.numpy()
X = data[mask].to_numpy(dtype=np.float32)
y = graph.y[mask].numpy().astype(np.int32)
t_all=graph.timestep[mask].numpy().astype(np.int16)

train_idx = t_all <= 28
val_idx = (t_all >= 29) & (t_all <= 34)
test_idx = t_all >= 35

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Calculate scale_pos_weight strictly on this isolated training block
n_licit = np.sum(y_train == 0)
n_illicit = np.sum(y_train == 1)

del data, graph
gc.collect()


30

In [3]:
params = {
'objective': 'binary',
'scale_pos_weight':n_licit/n_illicit,

'boosting_type': 'gbdt',

'learning_rate': 0.02,
'n_estimators': 5000,

'num_leaves': 31,
'max_depth': 6,

'min_child_samples': 50,

'subsample': 0.9,
'subsample_freq': 1,

'colsample_bytree': 0.4,

'reg_alpha': 0.1,
'reg_lambda': 1.0,

'random_state': 42,
'n_jobs': 1,

'verbosity': -1
}
callbacks = [
        early_stopping(
            stopping_rounds=100,
            verbose=True
        ),
        log_evaluation(period=200)
        ]

In [5]:
model = LGBMClassifier(**params)

# ONE single fit call. LightGBM handles the internal epoch iterations.
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=callbacks
)

# =====================================================================
# FINAL EVALUATION ON UNSEEN FUTURE TEST SET (X_test)
# =====================================================================
print("\n======================================================================================================")
print("Evaluating on Unseen Test Set (Timesteps 35-49)...")

test_preds_proba = model.predict_proba(X_test.copy(), num_iteration=model.best_iteration_)[:, 1]
test_preds_hard = (test_preds_proba >= 0.50).astype(int)

# Calculate final metrics
final_prauc = average_precision_score(y_test, test_preds_proba)

from sklearn.metrics import f1_score, classification_report
final_f1 = f1_score(y_test, test_preds_hard, pos_label=1)

print(f"Final Leak-Proof PR-AUC   : {final_prauc:.4f}")
print(f"Final Leak-Proof F1-Score : {final_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds_hard, target_names=['Licit', 'Illicit']))

X_test=pd.DataFrame(X_test)
y_test=pd.DataFrame(y_test)
model.booster_.save_model('/Users/rageshwer/Goal ML/Projects/AML - GNN/req_data/model.txt')
X_test.to_parquet('/Users/rageshwer/Goal ML/Projects/AML - GNN/req_data/test_vectors.parquet')
y_test.to_parquet('/Users/rageshwer/Goal ML/Projects/AML - GNN/req_data/test_truth.parquet')
# Clean up at the very end of the script
del model, X_train, X_val, X_test, y_train, y_val, y_test
gc.collect()

Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.99932	valid_0's binary_logloss: 0.0459174
[400]	valid_0's auc: 0.999479	valid_0's binary_logloss: 0.0264014
[600]	valid_0's auc: 0.999521	valid_0's binary_logloss: 0.0213209
[800]	valid_0's auc: 0.99954	valid_0's binary_logloss: 0.0202006
[1000]	valid_0's auc: 0.999546	valid_0's binary_logloss: 0.0200602
Early stopping, best iteration is:
[1041]	valid_0's auc: 0.999547	valid_0's binary_logloss: 0.0200329

Evaluating on Unseen Test Set (Timesteps 35-49)...
Final Leak-Proof PR-AUC   : 0.7158
Final Leak-Proof F1-Score : 0.6932

Classification Report:
              precision    recall  f1-score   support

       Licit       0.97      0.99      0.98     15587
     Illicit       0.87      0.58      0.69      1083

    accuracy                           0.97     16670
   macro avg       0.92      0.79      0.84     16670
weighted avg       0.96      0.97      0.96     16670



11